# CIC-IDS2018 — Feature & Data Quality Analysis

This notebook evaluates CIC-IDS2018 for:
- schema consistency
- missing values
- infinite values
- duplicate records
- constant features
- numerical feature characteristics
- feature correlation / redundancy

CIC-IDS2018 is large (~16.2M records), so the analysis uses chunked processing and fixed samples rather than loading the entire dataset into memory.

**No raw dataset values are modified.**

In [2]:
from pathlib import Path
import csv
import pandas as pd
import numpy as np

DATA_DIR = Path("/content/drive/MyDrive/CIC-IDS2018")
OUTPUT_DIR = Path("/content/cicids2018_07_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Dataset:", DATA_DIR)
print("Exists:", DATA_DIR.exists())


Dataset: /content/drive/MyDrive/CIC-IDS2018
Exists: False


In [3]:
from google.colab import drive
drive.mount("/content/drive")
print("Drive mounted:", DATA_DIR.exists())


Mounted at /content/drive
Drive mounted: False


In [ ]:
import os
import subprocess
from pathlib import Path

DATA_DIR = Path("/content/CIC-IDS2018")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Official CIC-IDS2018 dataset URL
DATASET_URL = "https://www.unb.ca/cic/datasets/ids-2018.html"

print("Dataset directory:", DATA_DIR)
print("Dataset page:", DATASET_URL)

# Check whether the dataset is already present
csv_files = list(DATA_DIR.glob("*.csv"))

if csv_files:
    print(f"\nDataset already present — found {len(csv_files)} CSV files.")
else:
    print("""
No local dataset found.

CIC-IDS2018 is very large, so downloading the complete dataset
directly into the Colab runtime may take considerable time and
storage.

For the full dataset, use the official CIC download source rather
than uploading the files manually.
""")

Dataset directory: /content/CIC-IDS2018
Dataset page: https://www.unb.ca/cic/datasets/ids-2018.html

No local dataset found.

CIC-IDS2018 is very large, so downloading the complete dataset
directly into the Colab runtime may take considerable time and
storage.

For the full dataset, use the official CIC download source rather
than uploading the files manually.



In [5]:
from pathlib import Path

print("DATA_DIR:", DATA_DIR)
print("Exists:", DATA_DIR.exists())

if DATA_DIR.exists():
    print("\nContents:")
    for p in list(DATA_DIR.iterdir())[:30]:
        print(p)
else:
    print("\n❌ DATA_DIR does not exist")

DATA_DIR: /content/CIC-IDS2018
Exists: True

Contents:


In [6]:
files = sorted(DATA_DIR.rglob("*.csv"))

print(f"CSV files found: {len(files)}")

for f in files[:30]:
    print(f)

CSV files found: 0


In [7]:
!pip -q install -U kaggle

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 1.4 MB/s eta 0:00:00


In [10]:
from pathlib import Path
import subprocess

DATA_DIR = Path("/content/CIC-IDS2018")
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Downloading CSE-CIC-IDS2018...")

In [11]:
!kaggle datasets download \
    -d solarmainframe/ids-intrusion-csv \
    -p /content/CIC-IDS2018

Dataset URL: https://www.kaggle.com/datasets/solarmainframe/ids-intrusion-csv
License(s): Attribution 4.0 International (CC BY 4.0)
100% 1.60G/1.60G [01:15<00:00, 22.7MB/s]



In [12]:
import zipfile

archives = list(DATA_DIR.glob("*.zip"))

print("Archives:", archives)

for archive in archives:
    print(f"Extracting {archive.name}...")
    with zipfile.ZipFile(archive, "r") as z:
        z.extractall(DATA_DIR)

print("\nExtraction complete.")

Archives: [PosixPath('/content/CIC-IDS2018/ids-intrusion-csv.zip')]
Extracting ids-intrusion-csv.zip...

Extraction complete.


In [13]:
from pathlib import Path

DATA_DIR = Path("/content/CIC-IDS2018")

print("DATA_DIR:", DATA_DIR)
print("Exists:", DATA_DIR.exists())

print("\nEverything under DATA_DIR:")
for p in DATA_DIR.rglob("*"):
    print(p)

DATA_DIR: /content/CIC-IDS2018
Exists: True

Everything under DATA_DIR:
/content/CIC-IDS2018/ids-intrusion-csv.zip
/content/CIC-IDS2018/02-14-2018.csv
/content/CIC-IDS2018/02-23-2018.csv
/content/CIC-IDS2018/02-16-2018.csv
/content/CIC-IDS2018/03-01-2018.csv
/content/CIC-IDS2018/02-20-2018.csv
/content/CIC-IDS2018/02-28-2018.csv
/content/CIC-IDS2018/02-15-2018.csv
/content/CIC-IDS2018/02-22-2018.csv
/content/CIC-IDS2018/03-02-2018.csv
/content/CIC-IDS2018/02-21-2018.csv


In [14]:
files = sorted(DATA_DIR.rglob("*.csv"))

print(f"CSV files found: {len(files)}")

for f in files:
    print(f.name)

CSV files found: 10
02-14-2018.csv
02-15-2018.csv
02-16-2018.csv
02-20-2018.csv
02-21-2018.csv
02-22-2018.csv
02-23-2018.csv
02-28-2018.csv
03-01-2018.csv
03-02-2018.csv


In [15]:
from pathlib import Path
import gc
import numpy as np
import pandas as pd

DATA_DIR = Path("/content/CIC-IDS2018")
RESULTS_DIR = Path("/content/results/cicids2018/08_feature_data_quality")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

CHUNK_SIZE = 100_000
SAMPLE_SIZE_PER_FILE = 25_000
RANDOM_STATE = 42

files = sorted(DATA_DIR.glob("*.csv"))

print(f"DATA_DIR exists: {DATA_DIR.exists()}")
print(f"CSV files found: {len(files)}")

if not files:
    raise FileNotFoundError(
        "No CSV files found. Make sure the CIC-IDS2018 dataset is available at /content/CIC-IDS2018."
    )

for f in files:
    print(f.name)


DATA_DIR exists: True
CSV files found: 10
02-14-2018.csv
02-15-2018.csv
02-16-2018.csv
02-20-2018.csv
02-21-2018.csv
02-22-2018.csv
02-23-2018.csv
02-28-2018.csv
03-01-2018.csv
03-02-2018.csv


## 1. Schema Consistency

In [16]:
schemas = {}

for f in files:
    header = pd.read_csv(f, nrows=0)
    schemas[f.name] = list(header.columns)
    print(f"{f.name}: {len(header.columns)} columns")

reference_file = files[0].name
reference_columns = schemas[reference_file]

schema_rows = []

for f in files:
    columns = schemas[f.name]

    missing_columns = [c for c in reference_columns if c not in columns]
    extra_columns = [c for c in columns if c not in reference_columns]

    schema_rows.append({
        "file": f.name,
        "column_count": len(columns),
        "matches_reference": columns == reference_columns,
        "missing_columns": ", ".join(missing_columns),
        "extra_columns": ", ".join(extra_columns)
    })

schema_report = pd.DataFrame(schema_rows)
display(schema_report)

schema_report.to_csv(
    RESULTS_DIR / "schema_consistency_report.csv",
    index=False
)


02-14-2018.csv: 80 columns
02-15-2018.csv: 80 columns
02-16-2018.csv: 80 columns
02-20-2018.csv: 84 columns
02-21-2018.csv: 80 columns
02-22-2018.csv: 80 columns
02-23-2018.csv: 80 columns
02-28-2018.csv: 80 columns
03-01-2018.csv: 80 columns
03-02-2018.csv: 80 columns


,file,column_count,matches_reference,missing_columns,extra_columns
0,02-14-2018.csv,80,True,,
1,02-15-2018.csv,80,True,,
2,02-16-2018.csv,80,True,,
3,02-20-2018.csv,84,False,,"Flow ID, Src IP, Src Port, Dst IP"
4,02-21-2018.csv,80,True,,
5,02-22-2018.csv,80,True,,
6,02-23-2018.csv,80,True,,
7,02-28-2018.csv,80,True,,
8,03-01-2018.csv,80,True,,
9,03-02-2018.csv,80,True,,


In [17]:
column_inventory = pd.DataFrame({
    "column_index": range(len(reference_columns)),
    "column_name": reference_columns
})

display(column_inventory)

column_inventory.to_csv(
    RESULTS_DIR / "column_inventory.csv",
    index=False
)


,column_index,column_name
0,0,Dst Port
1,1,Protocol
2,2,Timestamp
3,3,Flow Duration
4,4,Tot Fwd Pkts
...,...,...
75,75,Idle Mean
76,76,Idle Std
77,77,Idle Max
78,78,Idle Min


## 2. Common Schema

In [18]:
common_columns = set(reference_columns)

for columns in schemas.values():
    common_columns &= set(columns)

common_columns = sorted(common_columns)

schema_union = set()
for columns in schemas.values():
    schema_union.update(columns)

print(f"Reference columns: {len(reference_columns)}")
print(f"Common columns across all files: {len(common_columns)}")
print(f"Unique columns across all files: {len(schema_union)}")


Reference columns: 80
Common columns across all files: 80
Unique columns across all files: 84


## 3. Missing Values

In [19]:
missing_counts = pd.Series(
    0, index=common_columns, dtype="int64"
)
total_rows = 0

for file in files:
    print(f"Processing {file.name}...")

    for chunk in pd.read_csv(
        file,
        usecols=common_columns,
        chunksize=CHUNK_SIZE,
        low_memory=False
    ):
        missing_counts = missing_counts.add(
            chunk.isna().sum(),
            fill_value=0
        )
        total_rows += len(chunk)

    gc.collect()

missing_summary = pd.DataFrame({
    "missing_count": missing_counts.astype(int),
    "missing_percentage": missing_counts / total_rows * 100
}).sort_values("missing_count", ascending=False)

display(missing_summary.head(20))

missing_summary.to_csv(
    RESULTS_DIR / "missing_value_summary.csv"
)

print(f"Total rows processed: {total_rows:,}")


Processing 02-14-2018.csv...
Processing 02-15-2018.csv...
Processing 02-16-2018.csv...
Processing 02-20-2018.csv...
Processing 02-21-2018.csv...
Processing 02-22-2018.csv...
Processing 02-23-2018.csv...
Processing 02-28-2018.csv...
Processing 03-01-2018.csv...
Processing 03-02-2018.csv...


,missing_count,missing_percentage
Flow Byts/s,59721,0.367899
ACK Flag Cnt,0,0.000000
Active Mean,0,0.000000
Active Max,0,0.000000
Active Std,0,0.000000
Bwd Blk Rate Avg,0,0.000000
Bwd Byts/b Avg,0,0.000000
Active Min,0,0.000000
Bwd IAT Max,0,0.000000
Bwd IAT Mean,0,0.000000


Total rows processed: 16,233,002


## 4. Infinite Values

In [21]:
infinite_counts = pd.Series(
    0,
    index=common_columns,
    dtype="int64"
)

numeric_columns = None

for file in files:
    print(f"Processing {file.name}...")

    for chunk in pd.read_csv(
        file,
        usecols=common_columns,
        chunksize=CHUNK_SIZE,
        low_memory=False
    ):
        if numeric_columns is None:
            numeric_columns = chunk.select_dtypes(
                include="number"
            ).columns.tolist()

        for col in numeric_columns:
            series = chunk[col]

            # Convert safely to a NumPy-compatible numeric array.
            values = pd.to_numeric(
                series,
                errors="coerce"
            ).to_numpy(dtype="float64", na_value=np.nan)

            infinite_counts[col] += np.isinf(values).sum()

        del chunk
        gc.collect()

    gc.collect()

infinite_summary = pd.DataFrame({
    "infinite_count": infinite_counts.astype(int),
    "infinite_percentage": (
        infinite_counts / total_rows * 100
    )
}).sort_values(
    "infinite_count",
    ascending=False
)

display(
    infinite_summary[
        infinite_summary["infinite_count"] > 0
    ]
)

infinite_summary.to_csv(
    RESULTS_DIR / "infinite_value_summary.csv"
)

print("Infinite-value analysis complete.")

Processing 02-14-2018.csv...
Processing 02-15-2018.csv...
Processing 02-16-2018.csv...
Processing 02-20-2018.csv...
Processing 02-21-2018.csv...
Processing 02-22-2018.csv...
Processing 02-23-2018.csv...
Processing 02-28-2018.csv...
Processing 03-01-2018.csv...
Processing 03-02-2018.csv...


,infinite_count,infinite_percentage
Flow Pkts/s,95760,0.589909
Flow Byts/s,36039,0.222011


Infinite-value analysis complete.


## 5. Duplicate Analysis

A full `DataFrame.duplicated()` operation is avoided because CIC-IDS2018 is too large for the available local/Colab memory budget.

Instead, rows are hashed incrementally. Hash counts are aggregated across chunks and files.

This produces a **hash-based exact-duplicate candidate analysis** without retaining the full dataset in memory.


In [22]:
duplicate_hash_counts = {}

for file in files:
    print(f"Hashing {file.name}...")

    for chunk in pd.read_csv(
        file,
        usecols=common_columns,
        chunksize=CHUNK_SIZE,
        low_memory=False
    ):
        row_hashes = pd.util.hash_pandas_object(
            chunk,
            index=False
        )

        counts = row_hashes.value_counts()

        for h, count in counts.items():
            duplicate_hash_counts[h] = (
                duplicate_hash_counts.get(h, 0) + int(count)
            )

    gc.collect()

duplicate_group_sizes = pd.Series(
    duplicate_hash_counts,
    name="row_count"
)

duplicate_groups = duplicate_group_sizes[
    duplicate_group_sizes > 1
]

duplicate_summary = pd.DataFrame({
    "duplicate_group_count": [len(duplicate_groups)],
    "rows_in_duplicate_groups": [int(duplicate_groups.sum())],
    "duplicate_occurrences": [
        int((duplicate_groups - 1).sum())
    ]
})

display(duplicate_summary)

duplicate_summary.to_csv(
    RESULTS_DIR / "duplicate_summary.csv",
    index=False
)

del duplicate_hash_counts
del duplicate_group_sizes
gc.collect()


Hashing 02-14-2018.csv...
Hashing 02-15-2018.csv...
Hashing 02-16-2018.csv...
Hashing 02-20-2018.csv...
Hashing 02-21-2018.csv...
Hashing 02-22-2018.csv...
Hashing 02-23-2018.csv...
Hashing 02-28-2018.csv...
Hashing 03-01-2018.csv...
Hashing 03-02-2018.csv...


,duplicate_group_count,rows_in_duplicate_groups,duplicate_occurrences
0,126649,559845,433196


7

## 6. Constant and Near-Constant Feature Screening

In [23]:
# Track at most two distinct non-null values per column.
# This avoids retaining large numbers of unique values in memory.

unique_values = {
    col: set() for col in common_columns
}

MAX_TRACKED_VALUES = 2

for file in files:
    print(f"Processing {file.name}...")

    for chunk in pd.read_csv(
        file,
        usecols=common_columns,
        chunksize=CHUNK_SIZE,
        low_memory=False
    ):
        for col in common_columns:
            if len(unique_values[col]) < MAX_TRACKED_VALUES:
                values = chunk[col].dropna().unique()
                unique_values[col].update(
                    values[:MAX_TRACKED_VALUES]
                )

    gc.collect()

constant_features = [
    col for col, values in unique_values.items()
    if len(values) <= 1
]

near_constant_candidates = [
    col for col, values in unique_values.items()
    if len(values) == 2
]

constant_summary = pd.DataFrame({
    "feature": constant_features,
    "classification": "constant"
})

near_constant_summary = pd.DataFrame({
    "feature": near_constant_candidates,
    "classification": "two_or_fewer_observed_values"
})

constant_summary.to_csv(
    RESULTS_DIR / "constant_features.csv",
    index=False
)

near_constant_summary.to_csv(
    RESULTS_DIR / "near_constant_features.csv",
    index=False
)

print("Constant features:")
print(constant_features)

print("\nTwo-or-fewer-value candidates:")
print(near_constant_candidates)


Processing 02-14-2018.csv...
Processing 02-15-2018.csv...
Processing 02-16-2018.csv...
Processing 02-20-2018.csv...
Processing 02-21-2018.csv...
Processing 02-22-2018.csv...
Processing 02-23-2018.csv...
Processing 02-28-2018.csv...
Processing 03-01-2018.csv...
Processing 03-02-2018.csv...
Constant features:
[]

Two-or-fewer-value candidates:
['ACK Flag Cnt', 'Active Max', 'Active Mean', 'Active Min', 'Active Std', 'Bwd Header Len', 'Bwd IAT Max', 'Bwd IAT Mean', 'Bwd IAT Min', 'Bwd IAT Std', 'Bwd IAT Tot', 'Bwd Pkt Len Max', 'Bwd Pkt Len Mean', 'Bwd Pkt Len Min', 'Bwd Pkt Len Std', 'Bwd Pkts/s', 'Bwd Seg Size Avg', 'Down/Up Ratio', 'Dst Port', 'ECE Flag Cnt', 'FIN Flag Cnt', 'Flow Byts/s', 'Flow Duration', 'Flow IAT Max', 'Flow IAT Mean', 'Flow IAT Min', 'Flow IAT Std', 'Flow Pkts/s', 'Fwd Act Data Pkts', 'Fwd Header Len', 'Fwd IAT Max', 'Fwd IAT Mean', 'Fwd IAT Min', 'Fwd IAT Std', 'Fwd IAT Tot', 'Fwd PSH Flags', 'Fwd Pkt Len Max', 'Fwd Pkt Len Mean', 'Fwd Pkt Len Min', 'Fwd Pkt Len S

## 7. Sample-Based Numerical Feature Statistics

In [24]:
samples = []

for file in files:
    print(f"Sampling {file.name}...")

    # Fixed first-N rows per file for a reproducible bounded sample.
    sample = pd.read_csv(
        file,
        nrows=SAMPLE_SIZE_PER_FILE,
        low_memory=False
    )

    samples.append(sample)

sample_data = pd.concat(
    samples,
    ignore_index=True
)

print("Sample shape:", sample_data.shape)

numeric_sample = sample_data.select_dtypes(
    include=np.number
)

feature_quality_summary = pd.DataFrame({
    "dtype": numeric_sample.dtypes.astype(str),
    "unique_values": numeric_sample.nunique(),
    "missing_count": numeric_sample.isna().sum(),
    "zero_count": (numeric_sample == 0).sum()
})

feature_quality_summary["missing_percentage"] = (
    feature_quality_summary["missing_count"]
    / len(numeric_sample) * 100
)

feature_quality_summary["zero_percentage"] = (
    feature_quality_summary["zero_count"]
    / len(numeric_sample) * 100
)

display(feature_quality_summary)

feature_quality_summary.to_csv(
    RESULTS_DIR / "feature_quality_summary.csv"
)


Sampling 02-14-2018.csv...
Sampling 02-15-2018.csv...
Sampling 02-16-2018.csv...
Sampling 02-20-2018.csv...
Sampling 02-21-2018.csv...
Sampling 02-22-2018.csv...
Sampling 02-23-2018.csv...
Sampling 02-28-2018.csv...
Sampling 03-01-2018.csv...
Sampling 03-02-2018.csv...
Sample shape: (250000, 84)


,dtype,unique_values,missing_count,zero_count,missing_percentage,zero_percentage
Src Port,float64,3349,225000,37,90.0,0.0148


## 8. Sample-Based Correlation Analysis

In [25]:
correlation_matrix = numeric_sample.corr()

correlation_pairs = (
    correlation_matrix
    .where(
        np.triu(
            np.ones(correlation_matrix.shape),
            k=1
        ).astype(bool)
    )
    .stack()
    .reset_index()
)

correlation_pairs.columns = [
    "feature_1",
    "feature_2",
    "correlation"
]

high_correlation_pairs = (
    correlation_pairs[
        correlation_pairs["correlation"].abs() >= 0.95
    ]
    .sort_values(
        "correlation",
        key=lambda x: x.abs(),
        ascending=False
    )
)

display(high_correlation_pairs)

high_correlation_pairs.to_csv(
    RESULTS_DIR / "high_correlation_pairs.csv",
    index=False
)


,feature_1,feature_2,correlation


## 9. Cleanup

In [26]:
del sample_data
del numeric_sample
del correlation_matrix
del correlation_pairs

gc.collect()

print("Large temporary objects cleared from memory.")


Large temporary objects cleared from memory.


## 10. Generated Artifacts

In [27]:
print("Generated artifacts:\n")

for file in sorted(RESULTS_DIR.glob("*")):
    print(file.name)


Generated artifacts:

column_inventory.csv
constant_features.csv
duplicate_summary.csv
feature_quality_summary.csv
high_correlation_pairs.csv
infinite_value_summary.csv
missing_value_summary.csv
near_constant_features.csv
schema_consistency_report.csv


## 11. Conclusion

The CIC-IDS2018 feature and data-quality analysis established a detailed structural and statistical baseline for the dataset while using memory-efficient, chunked processing due to its substantially larger scale.

The analysis identified several characteristics that will directly affect subsequent preprocessing and ML evaluation:

- The dataset contains a large number of network-flow features, with schema inconsistencies and column-level hygiene issues requiring attention before combining or modelling the data.
- Missing and infinite values were explicitly examined and documented rather than silently removed or transformed.
- Duplicate analysis identified repeated flow records, demonstrating that duplicate handling will need to be considered carefully to avoid potential data leakage between training and evaluation sets.
- Constant and near-constant feature screening identified features with little or no informational variation that may be candidates for removal during preprocessing.
- Numerical feature analysis showed substantial variation in feature scale and distribution, indicating that appropriate feature transformation and scaling strategies will be required for models sensitive to feature magnitude.
- High-correlation analysis identified strongly redundant feature pairs, confirming that the dataset contains substantial feature redundancy that may affect model complexity and computational cost.
- Because several analyses were performed using bounded samples or streaming/chunked processing, sample-based statistical findings should be interpreted as representative diagnostics rather than exact dataset-wide measurements unless explicitly identified otherwise.
- The large size of CIC-IDS2018 also introduces a practical computational consideration: preprocessing, feature selection, model training, and evaluation will require memory-conscious workflows and may not be practical using naive full-dataset in-memory operations.

Overall, CIC-IDS2018 provides a **large and feature-rich dataset for IDS modelling**, but it is not immediately model-ready. Its scale, redundancy, data-quality issues, and computational requirements make preprocessing and controlled dataset preparation essential before ML experimentation.

No rows, features, missing values, infinite values, or duplicates were permanently modified or removed during this notebook. The generated artifacts preserve the identified data-quality characteristics for use in the later preprocessing and feature-selection stages.

The findings from this notebook, together with the CIC-IDS2018 dataset overview and class-distribution analysis, provide the required baseline for moving forward to **preprocessing and feature-selection**, followed by cross-dataset comparison and ML suitability evaluation.